# Erodep-1Myr run — post-processing & visualisation

This notebook turns a finished **goSPL** run into figures using the packaged
post-processing API (`gospl.analyse`) plus **PyGMT** for the global maps:

1. **Rasterise** each output step to a regular lon/lat **NetCDF grid** (`gospl.analyse.gridexport`).
2. **Global maps** of elevation and erosion/deposition (PyGMT).
3. **Per-basin river-mouth fluxes** — water & sediment discharge at every catchment outlet (`gospl.analyse.catchment`).
4. **Hypsometry / hypsographic curve** of the simulated topography.

Every gridded product shares one set of variable names, written by
`gridexport.to_netcdf` and read back by every downstream tool:
`elev`, `erodep`, `FA` (water discharge), `sedLoad` (sediment load),
`basin` (drainage-basin id), `chi`, `drainage_area`, on `lon`/`lat` axes.

## 1. Imports

- `gospl.analyse.gridexport` — rasterises a goSPL surface to a CF-NetCDF grid
  (fields + D8 drainage **basins** / **chi** / drainage area) and extracts
  per-basin river profiles.
- `gospl.analyse.catchment` — per-basin **outflow** points (cell of maximum
  water discharge / sediment load) from those grids.
- `gospl.analyse.stratasection` — stratigraphic cross-sections, wells & Wheeler
  diagrams (imported for convenience).
- `pygmt` for the global maps; `joblib` to grid several steps in parallel.

In [ ]:
import os
import glob
import pygmt
import numpy as np
import xarray as xr
import pandas as pd

import matplotlib
from matplotlib import cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1 import make_axes_locatable
%matplotlib inline

from joblib import Parallel, delayed

# --- gridded NetCDF + per-basin river profiles (PyGMT/ArcGIS + matplotlib) ---
from gospl.analyse.gridexport import (grid_export, to_netcdf)

# --- per-basin outflow fluxes (max water discharge / sediment load) ---
from gospl.analyse.catchment import basin_outflow, catchment_flux

## 2. Export each step to a gridded NetCDF — `grid_export` + `to_netcdf`

`grid_export(h5dir, mesh, step, spacing, latlim)` reassembles the global mesh,
interpolates every surface field onto a regular lon/lat grid and runs a raster
D8 hydrology pass (basins, chi, drainage area). `to_netcdf(grid, path)` writes a
self-describing CF-NetCDF (every variable carries `units` + `long_name`; the
run's **sea level** is stored as the `sea_level` attribute/variable).

| option | meaning |
|---|---|
| `spacing` | grid resolution in degrees (here `reso = 0.1`); default = median mesh edge |
| `step` | output step to grid |
| `latlim` | crop `|latitude|` (drops the singular polar caps; `90` keeps everything) |
| `base_level` | coast/outlet elevation for basins + chi (default: the run's sea level) |
| `fields` | subset of fields to grid (default: all) |

`getOutputsParallel` maps `grid_export`+`to_netcdf` over the requested steps with
`joblib` (each grid is independent), writing `results/surface{step}.nc`.

In [ ]:
# Define output folder name for the simulation
out_path = 'results/'

if not os.path.exists(out_path):
    os.makedirs(out_path)

In [ ]:
h5dir = "silico_mardep/h5"
mesh = "vars_40/mesh.npz"
reso = 0.1

out_name = "surface"

def getOutputs(steps):

    # clear any stale .nc files first
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return
        
    for stp in steps:
        g = grid_export(h5dir, mesh, stp, spacing=reso, latlim=90)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)                       

    return

def getOutputsParallel(steps, n_workers=8):
    for f in glob.glob(os.path.join(out_path, f"{out_name}*.nc")):
        try:
            os.remove(f)
        except PermissionError:
            print(f"Still locked, close it first: {f}")
            return

    def process_step(stp):
        g = grid_export(h5dir, mesh, stp, spacing=reso, latlim=90)
        fname = os.path.join(out_path, f"{out_name}{stp}.nc")
        to_netcdf(g, fname)

    Parallel(n_jobs=n_workers)(delayed(process_step)(stp) for stp in steps)

steps = [1,5,10] #np.arange(11)
getOutputsParallel(steps, n_workers=3)

## 3. Open the gridded NetCDFs

Load the per-step grids with `xarray` for the PyGMT plots below
(`dataset.elev`, `dataset.erodep`, `dataset.FA`, `dataset.sedLoad`, `dataset.basin`, …).

In [ ]:
dataset1 = xr.open_dataset(out_path+'/surface1.nc')
dataset5 = xr.open_dataset(out_path+'/surface5.nc')
dataset10 = xr.open_dataset(out_path+'/surface10.nc')

## 4. Global elevation map (PyGMT)

`grdimage` of `dataset10.elev` with the `geo` colour map and a 0 m coastline
contour (`grdcontour`), shown in two projections (Robinson `N12c` and Hammer
`W6i`); the colour bar is annotated in metres.

In [ ]:
fig = pygmt.Figure()
# Plotting elevation
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="geo", series=[-10000, 10000])
    fig.basemap(region='d', projection='W6i', frame='afg')
    fig.grdimage(dataset10.elev, frame=False)
    # Add contour
    fig.grdcontour(
        levels=0.1,
        grid=dataset10.elev,
        limit=[0., 0.09],
    )
    fig.colorbar(position="jBC+o0c/-1.5c+w8c/0.3c+h",frame=["a2000", "x+lElevation", "y+lm"])
# Customising the font style
fig.text(text="Step 10", position="TL", font="8p,Helvetica-Bold,black") #, xshift="-0.75c")
fig.show(dpi=500, width=1000)

## 5. Erosion / deposition map (PyGMT)

`grdimage` of `dataset10.erodep` (cumulative erosion negative / deposition
positive, metres) with the `vik` diverging colour map and the 0 m elevation
coastline overlaid for context.

In [ ]:
fig = pygmt.Figure()
# Plotting elevation
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="vik", series=[-5000, 5000])
    fig.basemap(region='d', projection='W6i', frame='afg')
    fig.grdimage(dataset10.erodep, frame=False)
    # Add contour
    fig.grdcontour(
        levels=0.1,
        grid=dataset10.elev,
        limit=[0.,0.09],
    )
    fig.colorbar(position="jBC+o0c/-1.5c+w8c/0.3c+h",frame=["a1000", "x+lErosion/Deposition", "y+lm"])
# Customising the font style
fig.text(text="Step 10", position="TL", font="8p,Helvetica-Bold,black") #, xshift="-0.75c")
fig.show(dpi=500, width=1000)

## 6. Flexural isostasy map (PyGMT)

`grdimage` of `dataset10.flexIso` (cumulative flexural isostasy
positive, metres) with the `bam` diverging colour map and the 0 m elevation
coastline overlaid for context.

In [ ]:
fig = pygmt.Figure()
# Plotting elevation
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="bam", series=[-2000, 2000])
    fig.basemap(region='d', projection='W6i', frame='afg')
    fig.grdimage(dataset10.flexIso, frame=False)
    # Add contour
    fig.grdcontour(
        levels=0.1,
        grid=dataset10.elev,
        limit=[0.,0.09],
    )
    fig.colorbar(position="jBC+o0c/-1.5c+w8c/0.3c+h",frame=["a500", "x+lFlexural isostasy", "y+lm"])
# Customising the font style
fig.text(text="Step 10", position="TL", font="8p,Helvetica-Bold,black") #, xshift="-0.75c")
fig.show(dpi=500, width=1000)

## 7. Per-basin outflow fluxes — `catchment_flux`

For every drainage **basin** we extract two outflow points: the cell of maximum
**water discharge** (`FA`) and the cell of maximum **sediment load** (`sedLoad`)
— each catchment's river mouth and its flux. `catchment_flux` reads the
`surface*.nc` grids **directly** (same variable names as `gridexport`, so no
renaming step), and the per-basin maximum is a single vectorised grouped
arg-max, so a global 0.1° grid is processed in ~1 s per step (serial — no MPI).

It writes `flowsed/flow{time}.csv` and `flowsed/sed{time}.csv`
(columns `basin,lon,lat,val`; `val` in m³/yr); basins with `≤ min_cells`
(default 10) cells are skipped. From a terminal (after `pip install -e .`) the
equivalent is `gospl-catchment -i index.csv -o flowsed`, where `index.csv` lists
one `surface*.nc` per `time`.

`catchment_flux` returns a nested dict: the outer key is the time value from your index (e.g. 1, 5, 10), and each maps to {"flow": DataFrame, "sed": DataFrame}. Each DataFrame has columns basin, lon, lat, val.

In [ ]:
# Per-basin outflow points (max water discharge / sediment load) per step.
# Reads the surface*.nc grids directly (gridexport variable names), writing
# flowsed/flow{t}.csv and flowsed/sed{t}.csv (columns: basin, lon, lat, val).
index = pd.DataFrame({"time": steps,
                      "netcdf": [out_path + f"surface{s}.nc" for s in steps]})
out = catchment_flux(index, "flowsed")

## 8. Load the flux CSVs and keep the strongest outlets

```python
out = catchment_flux(index, "flowsed")

# --- one step ---
flowdf = out[10]["flow"]      # water-discharge outlet per basin at time=10
seddf  = out[10]["sed"]       # sediment-load   outlet per basin at time=10

flowdf.columns                # Index(['basin', 'lon', 'lat', 'val'])
flowdf["val"].values          # discharge (m3/yr) at each basin's outlet
flowdf[["lon", "lat"]].values # outlet coordinates

# what time steps are available
list(out.keys())              # [1, 5, 10]
```

Iterate over all steps:

```python
for time, dfs in out.items():
    flow, sed = dfs["flow"], dfs["sed"]
    print(time, "→", len(flow), "outlets, peak FA =", flow["val"].max())
```
A couple of common patterns for your flux maps:
```python
# top-200 strongest water outlets at a step (what cells 9–11 do)
top = out[10]["flow"].nlargest(200, "val")
rLon, rLat, rFA = top["lon"].values, top["lat"].values, np.log10(top["val"].values)

# track a river mouth through time. NOTE: basin IDs are NOT stable between
# steps (each step is gridded and labelled independently), so match a catchment
# by its OUTLET LOCATION, not by id:
def flux_near(df, lon0, lat0, radius=1.0):
    """Largest outlet flux within `radius` degrees of (lon0, lat0)."""
    d = np.hypot(df["lon"] - lon0, df["lat"] - lat0)
    sub = df[d <= radius]
    return sub["val"].max() if len(sub) else np.nan

loc = (-54.9, 7.7)   # a river mouth of interest (lon, lat)
series = {t: flux_near(dfs["sed"], *loc) for t, dfs in out.items()}
```

> **Basin IDs are per-step labels** — `gridexport` re-derives the D8 basins for each grid and numbers outlets in processing order, so `basin 84` at one step is not the same catchment at another. Track a catchment across time by **outlet location** (above), not by id.

Note the keys are the raw time values from the index (ints here because your inputSedFlow.csv has 1,5,10). If you'd rather not keep the result in memory, pass outdir="flowsed" and read the per-step CSVs back exactly as your notebook already does (pd.read_csv("flowsed/flow10.csv")) — same basin,lon,lat,val columns.

In [ ]:
topF = out[10]["flow"].nlargest(200, "val")
rLonF, rLatF, rFA = topF["lon"].values, topF["lat"].values, np.log10(topF["val"].values)

topS = out[10]["sed"].nlargest(200, "val")
rLonS, rLatS, rSed = topS["lon"].values, topS["lat"].values, np.log10(topS["val"].values)

## 9. Water- and sediment-flux maps (PyGMT)

A greyscale elevation backdrop (`grdimage` of `dataset.elev`) with the top-200
outlets plotted as circles sized and coloured by log flux — `devon` for water
discharge, `buda` for sediment load (both m³/yr, log scale).

In [ ]:
# Let read the initial dataset (this will be used to get the elevation on our plot)
step = 10
dataset = xr.open_dataset(out_path+'/surface'+str(step)+'.nc')

fig = pygmt.Figure()
# Background image
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="gray", series=[-6000, 6000])
    fig.basemap(region='d', projection='W6i', frame='afg')
    fig.grdimage(dataset.elev, frame=False) #shading='+a345+nt1+m0',
    
    fig.grdcontour(
        levels=0.1,
        grid=dataset.elev,
        limit=[0, 0.09],
    )
# Scatter plot
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="devon", series=[rFA.min(), rFA.max()], reverse=True)
    fig.plot(
        x=rLonF,
        y=rLatF,
        style="cc",
        pen="white",
        size=0.00005 * 2 ** rFA,
        fill=rFA,
        cmap=True,
    )
    fig.colorbar(position="jBC+o0c/-1.5c+w8c/0.3c+h", 
                 frame="af+l'Water fluxes (log-scale) (m3/yr)'")
# Time interval
fig.text(text="Step 10", position="TL", font="8p,Helvetica-Bold,black")

fig.show(dpi=500, width=1000)
# fname = 'fluxes/flow'+str(step)+'Ma.png'
# fig.savefig(fname=fname,dpi=500)

In [ ]:
fig = pygmt.Figure()

# Background image
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="gray", series=[-6000, 6000])
    fig.basemap(region='d', projection='W6i', frame='afg')
    fig.grdimage(dataset.elev, frame=False)
    fig.grdcontour(
        levels=0.1,
        grid=dataset.elev,
        limit=[0., 0.09],
    )
# Scatter plot
with pygmt.config(FONT='6p,Helvetica,black'):
    pygmt.makecpt(cmap="buda", series=[rSed.min(), rSed.max()])
    fig.plot(
        x=rLonS,
        y=rLatS,
        style="cc",
        pen="black",
        size=0.0005 * 2 ** rSed,
        fill=rSed,
        cmap=True,
    )
    fig.colorbar(position="jBC+o0c/-1.5c+w8c/0.3c+h", 
                 frame="af+l'Sediment fluxes (log-scale) (m3/yr)'")
# Time interval
fig.text(text="Step 10", position="TL", font="8p,Helvetica-Bold,black")
fig.show(dpi=500, width=1000)
# fname = 'fluxes/sed'+str(step)+'Ma.png'
# fig.savefig(fname=fname,dpi=500)